# Barcelona Traffic Data Generator — Fabric Conference 2026 (CCIB)

Generates a realistic, real-time traffic feed for Barcelona road segments from `segment_long.csv`, classified with the official `Speed_state.csv` and `status.csv` reference tables, and streams it to an **Azure Event Hub / Fabric Eventstream** endpoint.

**Event window:** Sun 27 Sep – Thu 1 Oct 2026 (Microsoft Fabric Conference, held at **CCIB**, Sant Martí district).

| Column | Description |
|---|---|
| `segment_id` | Authoritative road ID (`Tram`) from `segment_long.csv` |
| `road_name`, `start_lat/lng`, `end_lat/lng`, `mid_lat/lng` | Geometry derived from the ordered `Tram_Components` points |
| `zone_approx` | Zone assigned from segment geometry and `Descripci_`, not a mismatched ID lookup |
| `status_code` | From `status.csv` — 1 Active, 0 No Data (sensor outage) |
| `speed_state_code` | From `Speed_state.csv` — -1 No Data, 0 Unknown/Below Threshold, 1 Fluid, 2 Dense, 3 Congested |
| `avg_speed_kmh` | Sampled inside the `typical_speed_kmh` band of the assigned `speed_state_code` |
| `vehicle_count` | Derived from a zone-capacity × time-of-day × event-load model |
| `incident_flag` | True when congestion (`speed_state_code == 3`) is attributed to a probable incident, plus a handful of injected real-world incidents |
| `timestamp` | 5-minute cadence, Europe/Madrid (CEST, UTC+2) |

**Congestion modelling:** the zone `Sant Marti / Ronda Litoral` (where CCIB sits) and, to a lesser extent, `Eixample / Diagonal` (the main approach avenue) and the ring-road zones (`Ronda de Dalt / Nord`, `Accesses / Periphery`) receive elevated load during conference arrival, lunch and departure windows each day — producing realistic recurring congestion near the venue, on top of a handful of injected unplanned incidents.


In [ ]:
# Install the Azure Event Hubs SDK
%pip install azure-eventhub --quiet

In [1]:
import math
import random
import json
import time
import numpy as np
import pandas as pd
from datetime import datetime, timedelta, timezone, date
from azure.eventhub import EventHubProducerClient, EventData

In [ ]:
# ── Fabric Eventstream / Azure Event Hub connection ───────────────────────────
# Paste from: Fabric portal -> Eventstream -> Sources -> Custom endpoint
TRAFFIC_EH_CONN_STR = 'YOUR_EVENTHUB_CONNECTION_STRING_HERE'
TRAFFIC_EH_NAME     = 'YOUR_EVENTHUB_NAME_HERE'

In [ ]:
# ── Reference data files ──────────────────────────────────────────────────────
SEGMENT_LONG_CSV = '/lakehouse/default/Files/segment_long.csv'
SPEED_STATE_CSV = '/lakehouse/default/Files/speed_state.csv'
STATUS_CSV      = '/lakehouse/default/Files/status.csv'

# ── Event window (Europe/Madrid, CEST = UTC+2 in late Sep / early Oct) ───────
BCN_TZ      = timezone(timedelta(hours=2))
EVENT_START = date(2026, 9, 27)   # Sun - Partner Day / registration
EVENT_END   = date(2026, 10, 1)   # Thu - final day / departures

# ── Data cadence & playback compression for streaming ─────────────────────────
INTERVAL_MINUTES   = 5                                        # data granularity
SPEED_FACTOR       = 12                                       # 12x -> 5 min real = 25 s playback
EMIT_DELAY_SECONDS = (INTERVAL_MINUTES * 60) / SPEED_FACTOR


random.seed(42)
np.random.seed(42)
print(f'Event window: {EVENT_START} -> {EVENT_END} | interval={INTERVAL_MINUTES}min | '
      f'playback speed={SPEED_FACTOR}x (~{EMIT_DELAY_SECONDS:.1f}s per batch)')

Event window: 2026-09-27 -> 2026-10-01 | interval=5min | playback speed=12x (~25.0s per batch)


In [ ]:
# Load road geometry and official reference tables
segment_long_df = pd.read_csv(SEGMENT_LONG_CSV)
speed_state_df = pd.read_csv(SPEED_STATE_CSV)
status_df      = pd.read_csv(STATUS_CSV)

segment_long_df = segment_long_df.sort_values(['Tram', 'Tram_Components'])
segment_geometry = (
    segment_long_df.groupby('Tram', as_index=False)
    .agg(
        road_name=('Descripci_', 'first'),
        start_lng=('Longitud', 'first'), start_lat=('Latitud', 'first'),
        end_lng=('Longitud', 'last'), end_lat=('Latitud', 'last'),
        mid_lng=('Longitud', 'mean'), mid_lat=('Latitud', 'mean'),
        coordinate_count=('Tram_Components', 'count'),
    )
    .rename(columns={'Tram': 'segment_id'})
)

CCIB_LAT, CCIB_LNG = 41.4085, 2.2173

def haversine_m(lat1, lng1, lat2, lng2):
    radius_m = 6_371_000
    lat_delta = math.radians(lat2 - lat1)
    lng_delta = math.radians(lng2 - lng1)
    a = (math.sin(lat_delta / 2) ** 2 + math.cos(math.radians(lat1)) *
         math.cos(math.radians(lat2)) * math.sin(lng_delta / 2) ** 2)
    return radius_m * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

def assign_zone(segment):
    road = segment['road_name'].lower()
    distance_to_ccib_m = haversine_m(segment['mid_lat'], segment['mid_lng'], CCIB_LAT, CCIB_LNG)
    if distance_to_ccib_m <= 3_000:
        return 'Sant Marti / Ronda Litoral'
    if 'ronda de dalt' in road or segment['mid_lat'] >= 41.425:
        return 'Ronda de Dalt / Nord'
    if 'ronda litoral' in road or 'ronda del litoral' in road:
        return 'Accesses / Periphery'
    if 'diagonal' in road:
        return 'Eixample / Diagonal'
    if 'gran via' in road:
        return 'Eixample / Gran Via'
    if any(term in road for term in ('sants', 'carles iii', 'numància', 'numancia', 'entença')):
        return 'Les Corts / Sants'
    if any(term in road for term in ('sarrià', 'sarria', 'pedralbes', 'foix', 'ganduxer')):
        return 'Sarria-Sant Gervasi'
    if any(term in road for term in ('gràcia', 'gracia', 'via augusta', 'balmes', 'muntaner')):
        return 'Gracia / Via Augusta'
    return 'Eixample / Gran Via'

segment_geometry['distance_to_ccib_m'] = segment_geometry.apply(
    lambda row: haversine_m(row['mid_lat'], row['mid_lng'], CCIB_LAT, CCIB_LNG), axis=1
)
segment_geometry['zone_approx'] = segment_geometry.apply(assign_zone, axis=1)
segment_ids = segment_geometry['segment_id'].tolist()
segment_zone = dict(zip(segment_geometry['segment_id'], segment_geometry['zone_approx']))

# Sampling ranges inside each speed_state_code's typical_speed_kmh band
SPEED_BAND_SAMPLE = {
    1: (46, 68),   # Fluid: >45 km/h
    2: (25, 45),   # Dense: 25-45 km/h
    3: (4, 24),    # Congested: <25 km/h
}
STATUS_ACTIVE, STATUS_NO_DATA = 1, 0

print(f'{len(segment_ids)} real road segments loaded across {segment_geometry["zone_approx"].nunique()} zones')
display(segment_geometry.head())
display(speed_state_df)
display(status_df)

78 segments loaded across 8 zones


,speed_state_code,speed_state_label,description,typical_speed_kmh
0,-1,No Data,No sensor reading available for this period; s...,NaN
1,0,Unknown / Below Threshold,Section active but speed cannot be classified ...,NaN
2,1,Fluid,Traffic flowing freely,>45
3,2,Dense,Traffic flowing but with notable congestion,25–45
4,3,Congested,Heavy congestion; significantly reduced speeds,<25


,status_code,status_label,description
0,0,No Data,Sensor or section has no available reading for...
1,1,Active,Section is active and sensor data is available


In [4]:
# Typical peak capacity (vehicles / 5-min window) per zone
ZONE_CAPACITY = {
    'Eixample / Gran Via':          100,
    'Eixample / Diagonal':          110,
    'Les Corts / Sants':             90,
    'Sarria-Sant Gervasi':           70,
    'Gracia / Via Augusta':          75,
    'Sant Marti / Ronda Litoral':    95,
    'Ronda de Dalt / Nord':         160,
    'Accesses / Periphery':         130,
}

# How strongly each zone is pulled into CCIB conference traffic (0 = unaffected, 1 = maximum)
ZONE_CCIB_WEIGHT = {
    'Sant Marti / Ronda Litoral':  1.00,   # CCIB itself sits in this zone (Diagonal Mar / seafront ring road)
    'Eixample / Diagonal':         0.55,   # Avinguda Diagonal is the main approach from the city centre
    'Ronda de Dalt / Nord':        0.35,   # ring road used by out-of-town / airport traffic
    'Accesses / Periphery':        0.30,   # highway accesses feeding the ring roads
    'Eixample / Gran Via':         0.15,
    'Les Corts / Sants':           0.05,
    'Gracia / Via Augusta':        0.05,
    'Sarria-Sant Gervasi':         0.03,
}

def base_flow(dt):
    """Typical Barcelona weekday traffic curve (~0.2-1.4): morning, lunch and evening peaks, Sunday damped."""
    t = dt.hour + dt.minute / 60
    flow  = 0.20
    flow += 0.55 * math.exp(-0.5 * ((t - 8.5) / 1.1) ** 2)
    flow += 0.35 * math.exp(-0.5 * ((t - 14.0) / 1.0) ** 2)
    flow += 0.60 * math.exp(-0.5 * ((t - 18.5) / 1.3) ** 2)
    if dt.weekday() == 6:      # Sunday
        flow *= 0.55
    return flow

print('Zone capacity & CCIB-proximity weights defined.')

Zone capacity & CCIB-proximity weights defined.


In [5]:
# Fabric Conference daily schedule: (start_h, start_m, end_h, end_m, multiplier)
# Applied to CCIB-adjacent zones (scaled by ZONE_CCIB_WEIGHT), on top of the normal daily flow curve.
EVENT_WINDOWS = {
    date(2026, 9, 27): [(15, 0, 19, 0, 1.4)],                                              # Sun: registration / arrival
    date(2026, 9, 28): [(7, 30, 9, 30, 2.0), (12, 30, 14, 0, 1.5), (18, 0, 20, 0, 1.7)],    # Day 1: opening keynote
    date(2026, 9, 29): [(7, 30, 9, 30, 2.1), (12, 30, 14, 0, 1.6), (18, 0, 20, 0, 1.6)],    # Day 2: peak attendance
    date(2026, 9, 30): [(7, 30, 9, 30, 1.9), (12, 30, 14, 0, 1.5), (18, 0, 20, 0, 1.5)],    # Day 3
    date(2026, 10, 1): [(7, 30, 9, 0, 1.4), (13, 0, 17, 0, 2.0)],                           # Day 4: early departures
}

def event_multiplier(dt):
    """Base event multiplier active at dt (before zone weighting), or 1.0 outside conference windows."""
    windows = EVENT_WINDOWS.get(dt.date())
    if not windows:
        return 1.0
    t = dt.hour * 60 + dt.minute
    for sh, sm, eh, em, mult in windows:
        if sh * 60 + sm <= t < eh * 60 + em:
            return mult
    return 1.0

print('FabCon daily event windows defined.')

FabCon daily event windows defined.


In [6]:
# A handful of realistic, unplanned incidents (accident/breakdown) layered on top of the recurring
# rush-hour + event congestion. Weighted towards CCIB-adjacent zones, with a couple elsewhere in the city.
INCIDENT_SEGMENTS_POOL = [s for s in segment_ids if segment_zone[s] in
                          ('Sant Marti / Ronda Litoral', 'Eixample / Diagonal')] * 3 + segment_ids
_inc_rng = random.Random(7)

def _random_incident_start():
    d = _inc_rng.choice([date(2026, 9, 27), date(2026, 9, 28), date(2026, 9, 29),
                         date(2026, 9, 30), date(2026, 10, 1)])
    h = _inc_rng.randint(6, 21)
    m = _inc_rng.choice([0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55])
    return datetime(d.year, d.month, d.day, h, m, tzinfo=BCN_TZ)

INCIDENTS = []
for _ in range(7):
    start = _random_incident_start()
    duration = _inc_rng.randint(20, 55)
    INCIDENTS.append({
        'segment_id': _inc_rng.choice(INCIDENT_SEGMENTS_POOL),
        'start': start,
        'end': start + timedelta(minutes=duration),
    })

def active_incident(segment_id, dt):
    return any(inc['segment_id'] == segment_id and inc['start'] <= dt < inc['end'] for inc in INCIDENTS)

print(f'{len(INCIDENTS)} unplanned incidents scheduled:')
for inc in sorted(INCIDENTS, key=lambda i: i['start']):
    print(f"  segment {inc['segment_id']:>3} ({segment_zone[inc['segment_id']]}): "
          f"{inc['start'].strftime('%a %d %H:%M')} -> {inc['end'].strftime('%H:%M')}")

7 unplanned incidents scheduled:
  segment  42 (Gracia / Via Augusta): Sun 27 09:15 -> 09:38
  segment  55 (Sant Marti / Ronda Litoral): Sun 27 13:00 -> 13:55
  segment  49 (Gracia / Via Augusta): Sun 27 13:05 -> 14:00
  segment  48 (Gracia / Via Augusta): Mon 28 07:05 -> 07:52
  segment  59 (Sant Marti / Ronda Litoral): Tue 29 10:30 -> 10:53
  segment  51 (Sant Marti / Ronda Litoral): Tue 29 19:10 -> 20:04
  segment  70 (Ronda de Dalt / Nord): Thu 01 09:25 -> 09:48


In [ ]:
zone_list    = segment_geometry['zone_approx'].values
seg_id_arr   = segment_geometry['segment_id'].values
capacities   = np.array([ZONE_CAPACITY[z] for z in zone_list], dtype=float)
ccib_weights = np.array([ZONE_CCIB_WEIGHT[z] for z in zone_list], dtype=float)
n_seg        = len(seg_id_arr)

timestamps = pd.date_range(
    start=datetime(EVENT_START.year, EVENT_START.month, EVENT_START.day, tzinfo=BCN_TZ),
    end=datetime(EVENT_END.year, EVENT_END.month, EVENT_END.day, 23, 55, tzinfo=BCN_TZ),
    freq=f'{INTERVAL_MINUTES}min',
)

rows = []
for ts in timestamps:
    base = base_flow(ts)
    mult = event_multiplier(ts)
    effective_mult = 1 + (mult - 1) * ccib_weights
    load = base * effective_mult * np.random.normal(1.0, 0.12, n_seg)

    vehicle_count = np.round(capacities * load * np.random.normal(1.0, 0.10, n_seg)).clip(min=0).astype(int)

    speed_state_code = np.where(load >= 0.75, 3, np.where(load >= 0.40, 2, 1))

    # Deep-night, near-empty segments: sensor active but volume too low to classify a speed state
    night = ts.hour < 5
    below_threshold = night & (vehicle_count <= 1) & (np.random.random(n_seg) < 0.30)
    speed_state_code = np.where(below_threshold, 0, speed_state_code)

    # Occasional sensor outage
    no_data = np.random.random(n_seg) < 0.008
    status_code = np.where(no_data, STATUS_NO_DATA, STATUS_ACTIVE)
    speed_state_code = np.where(no_data, -1, speed_state_code)

    avg_speed = np.full(n_seg, np.nan)
    for code, (lo, hi) in SPEED_BAND_SAMPLE.items():
        mask = speed_state_code == code
        if mask.any():
            avg_speed[mask] = np.random.uniform(lo, hi, mask.sum())

    # incident_flag mostly follows congestion: likely cause when code==3, rare precursor otherwise
    roll = np.random.random(n_seg)
    incident_flag = np.where(
        speed_state_code == 3, roll < 0.65,
        np.where(np.isin(speed_state_code, [1, 2]), roll < 0.01, False),
    )

    # Scheduled unplanned incidents override the computed state for their segment/time window
    for i, seg in enumerate(seg_id_arr):
        if not no_data[i] and active_incident(seg, ts):
            speed_state_code[i] = 3
            incident_flag[i] = True
            avg_speed[i] = np.random.uniform(4, 15)
            vehicle_count[i] = int(capacities[i] * 0.30 * np.random.normal(1.0, 0.1))

    vehicle_count = np.where(no_data, 0, vehicle_count)
    avg_speed = np.where(no_data, np.nan, avg_speed)
    incident_flag = np.where(no_data, False, incident_flag)

    for i in range(n_seg):
        rows.append((
            seg_id_arr[i], ts.isoformat(), int(status_code[i]), int(speed_state_code[i]),
            round(float(avg_speed[i]), 1) if not np.isnan(avg_speed[i]) else None,
            int(vehicle_count[i]), bool(incident_flag[i]), zone_list[i],
            segment_geometry['road_name'].iloc[i],
            round(float(segment_geometry['start_lat'].iloc[i]), 6),
            round(float(segment_geometry['start_lng'].iloc[i]), 6),
            round(float(segment_geometry['end_lat'].iloc[i]), 6),
            round(float(segment_geometry['end_lng'].iloc[i]), 6),
            round(float(segment_geometry['mid_lat'].iloc[i]), 6),
            round(float(segment_geometry['mid_lng'].iloc[i]), 6),
            round(float(segment_geometry['distance_to_ccib_m'].iloc[i]), 1),
        ))

df_traffic = pd.DataFrame(rows, columns=[
    'segment_id', 'timestamp', 'status_code', 'speed_state_code',
    'avg_speed_kmh', 'vehicle_count', 'incident_flag', 'zone_approx',
    'road_name', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
    'mid_lat', 'mid_lng', 'distance_to_ccib_m',
])
print(f'Generated {len(df_traffic):,} records | {n_seg} segments | '
      f'{timestamps.min()} -> {timestamps.max()}')
df_traffic.head(10)

Generated 112,320 records | 78 segments | 2026-09-27 00:00:00+02:00 -> 2026-10-01 23:55:00+02:00


,segment_id,timestamp,status_code,speed_state_code,avg_speed_kmh,vehicle_count,incident_flag,zone_approx
0,1,2026-09-27T00:00:00+02:00,1,1,66.7,12,False,Eixample / Gran Via
1,2,2026-09-27T00:00:00+02:00,1,1,54.5,9,False,Eixample / Gran Via
2,3,2026-09-27T00:00:00+02:00,1,1,67.1,12,False,Eixample / Gran Via
3,4,2026-09-27T00:00:00+02:00,1,1,65.9,13,False,Eixample / Gran Via
4,5,2026-09-27T00:00:00+02:00,1,1,50.3,12,False,Eixample / Gran Via
5,6,2026-09-27T00:00:00+02:00,1,1,47.5,10,False,Eixample / Gran Via
6,7,2026-09-27T00:00:00+02:00,1,1,48.2,12,False,Eixample / Gran Via
7,8,2026-09-27T00:00:00+02:00,1,1,46.4,11,False,Eixample / Gran Via
8,9,2026-09-27T00:00:00+02:00,1,1,48.1,11,False,Eixample / Gran Via
9,10,2026-09-27T00:00:00+02:00,1,1,61.0,12,False,Eixample / Gran Via


In [8]:
# Sanity check: CCIB zone should show materially more congestion & incidents than the rest of the city
CCIB_ZONE = 'Sant Marti / Ronda Litoral'
ccib_df   = df_traffic[df_traffic['zone_approx'] == CCIB_ZONE]
other_df  = df_traffic[df_traffic['zone_approx'] != CCIB_ZONE]

print(f'Speed-state distribution — {CCIB_ZONE} (CCIB):')
print(ccib_df['speed_state_code'].value_counts(normalize=True).sort_index().round(3).to_string())
print('\nSpeed-state distribution — other zones:')
print(other_df['speed_state_code'].value_counts(normalize=True).sort_index().round(3).to_string())

print(f"\nIncident rate near CCIB : {ccib_df['incident_flag'].mean():.2%}")
print(f"Incident rate elsewhere : {other_df['incident_flag'].mean():.2%}")

peak = ccib_df[(pd.to_datetime(ccib_df['timestamp']).dt.date == date(2026, 9, 29)) &
               (pd.to_datetime(ccib_df['timestamp']).dt.hour.between(7, 9))]
print(f"\nDay-2 07:00-09:00 near CCIB -> avg speed {peak['avg_speed_kmh'].mean():.1f} km/h, "
      f"congested share: {(peak['speed_state_code'] == 3).mean():.1%}")

Speed-state distribution — Sant Marti / Ronda Litoral (CCIB):
speed_state_code
-1    0.008
 1    0.636
 2    0.196
 3    0.159

Speed-state distribution — other zones:
speed_state_code
-1    0.008
 1    0.658
 2    0.250
 3    0.084

Incident rate near CCIB : 11.52%
Incident rate elsewhere : 6.28%

Day-2 07:00-09:00 near CCIB -> avg speed 21.8 km/h, congested share: 66.4%


## Streaming to Fabric Eventstream

The cell below streams every 5-minute window (all real geometry segments) as one Event Hub batch, in chronological order, at compressed playback speed (`SPEED_FACTOR`).

Each event includes the `Tram`-based `segment_id`, `road_name`, assigned `zone_approx`, and segment start/end/midpoint coordinates for map rendering.

Set `LIVE_SLEEP = False` to bulk-send the whole dataset instantly (useful for backfill/testing). Requires `pip install azure-eventhub` and a valid connection string/name in the config cell above.

In [ ]:
STREAM_COLUMNS = ['segment_id', 'timestamp', 'status_code', 'speed_state_code',
                  'vehicle_count', 'avg_speed_kmh', 'incident_flag', 'zone_approx',
                  'road_name', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
                  'mid_lat', 'mid_lng', 'distance_to_ccib_m']
LIVE_SLEEP = True   # False = bulk send without delay

producer = EventHubProducerClient.from_connection_string(
    conn_str=TRAFFIC_EH_CONN_STR, eventhub_name=TRAFFIC_EH_NAME
)

total_sent = 0
with producer:
    for ts, grp in df_traffic.groupby('timestamp', sort=True):
        batch = producer.create_batch()
        for rec in grp[STREAM_COLUMNS].to_dict(orient='records'):
            batch.add(EventData(json.dumps(rec, default=str)))
        producer.send_batch(batch)
        total_sent += len(grp)
        n_congested = int((grp['speed_state_code'] == 3).sum())
        print(f'[{datetime.now().strftime("%H:%M:%S")}] ts={ts}  sent={len(grp):2d}  congested={n_congested:2d}')
        if LIVE_SLEEP:
            time.sleep(EMIT_DELAY_SECONDS)

print(f'\nDone. Total records sent to Eventstream: {total_sent:,}')

## 9. Eventhouse / KQL setup

Create `TrafficStream` with schema inference from Eventstream, or use this explicit schema:

`segment_id, timestamp, status_code, speed_state_code, vehicle_count, avg_speed_kmh, incident_flag, zone_approx, road_name, start_lat, start_lng, end_lat, end_lng, mid_lat, mid_lng, distance_to_ccib_m`

Zone and geometry are already present in each streamed record and are sourced from `segment_long.csv`, so the queries use those fields directly without a separate segment lookup.

- **`SpeedState`** ← `Speed_state.csv` (`speed_state_code`, `speed_state_label`, `description`, `typical_speed_kmh`)
- **`Status`** ← `status.csv` (`status_code`, `status_label`, `description`)

The cell below prints 10 ready-to-paste KQL queries covering live snapshots, zone congestion, the CCIB corridor, incident detection, rush-hour comparisons, anomaly detection and sensor health.

In [ ]:
KQL_QUERIES = {

'1. Live snapshot per segment (latest reading, most congested first)': """
TrafficStream
| summarize arg_max(timestamp, status_code, speed_state_code, avg_speed_kmh, vehicle_count, incident_flag, zone_approx) by segment_id
| lookup kind=leftouter (SpeedState | project speed_state_code, speed_state_label) on speed_state_code
| project timestamp, segment_id, zone_approx, speed_state_label, avg_speed_kmh, vehicle_count, incident_flag
| order by speed_state_label desc, avg_speed_kmh asc
""",

'2. Zone-level congestion summary (last 30 min)': """
TrafficStream
| where timestamp > ago(30m)
| summarize avg_speed_kmh    = round(avg(avg_speed_kmh), 1),
            total_vehicles   = sum(vehicle_count),
            pct_congested    = round(100.0 * countif(speed_state_code == 3) / count(), 1),
            incident_rate_pct = round(100.0 * countif(incident_flag) / count(), 1)
  by zone_approx
| order by pct_congested desc
""",

'3. CCIB corridor congestion trend (5-min bins)': """
TrafficStream
| where zone_approx == "Sant Marti / Ronda Litoral"
| summarize avg_speed_kmh = avg(avg_speed_kmh), total_vehicles = sum(vehicle_count) by bin(timestamp, 5m)
| render timechart
""",

'4. Top 10 most congested segments right now': """
TrafficStream
| summarize arg_max(timestamp, avg_speed_kmh, vehicle_count, speed_state_code, incident_flag, zone_approx) by segment_id
| where speed_state_code == 3
| top 10 by avg_speed_kmh asc
| project segment_id, zone_approx, avg_speed_kmh, vehicle_count, incident_flag
""",

'5. Active / recent incidents (last 2 hours)': """
TrafficStream
| where incident_flag == true and timestamp > ago(2h)
| summarize incident_start = min(timestamp), incident_end = max(timestamp),
            min_speed_kmh   = min(avg_speed_kmh), readings = count()
  by segment_id, zone_approx
| extend duration_min = datetime_diff('minute', incident_end, incident_start)
| order by incident_start desc
""",

'6. Conference rush-hour comparison by zone': """
TrafficStream
| extend hr = hourofday(timestamp)
| extend rush_window = case(hr between (7 .. 9), "Morning Arrival",
                             hr between (12 .. 14), "Lunch",
                             hr between (18 .. 20), "Evening Departure",
                             "Off-Peak")
| summarize avg_speed_kmh = round(avg(avg_speed_kmh), 1),
            pct_congested = round(100.0 * countif(speed_state_code == 3) / count(), 1)
  by zone_approx, rush_window
| order by zone_approx asc, rush_window asc
""",

'7. Anomaly detection on CCIB vehicle volume': """
TrafficStream
| where zone_approx == "Sant Marti / Ronda Litoral"
| make-series total_vehicles = sum(vehicle_count) on timestamp step 5m
| extend anomalies = series_decompose_anomalies(total_vehicles, 1.5)
| render anomalychart with (anomalycolumns=anomalies)
""",

'8. Sensor / data-quality health check (last hour)': """
TrafficStream
| where timestamp > ago(1h)
| summarize total = count(), no_data = countif(status_code == 0) by segment_id
| extend no_data_pct = round(100.0 * no_data / total, 1)
| where no_data_pct > 0
| order by no_data_pct desc
""",

'9. Speed-state distribution across the whole event': """
TrafficStream
| lookup kind=leftouter (SpeedState | project speed_state_code, speed_state_label) on speed_state_code
| summarize count() by speed_state_label
| render piechart
""",

'10. Day-by-day CCIB corridor congestion trend': """
TrafficStream
| where zone_approx in ("Sant Marti / Ronda Litoral", "Eixample / Diagonal")
| summarize pct_congested = round(100.0 * countif(speed_state_code == 3) / count(), 1),
            avg_speed_kmh = round(avg(avg_speed_kmh), 1)
  by startofday(timestamp)
| order by timestamp asc
| render columnchart
""",

}

for title, kql in KQL_QUERIES.items():
    print(f'=== {title} ===')
    print(kql.strip())
    print()

=== 1. Live snapshot per segment (latest reading, most congested first) ===
TrafficStream
| summarize arg_max(timestamp, status_code, speed_state_code, avg_speed_kmh, vehicle_count, incident_flag) by segment_id
| lookup kind=leftouter (Segment) on segment_id
| lookup kind=leftouter (SpeedState | project speed_state_code, speed_state_label) on speed_state_code
| project timestamp, segment_id, zone_approx, speed_state_label, avg_speed_kmh, vehicle_count, incident_flag
| order by speed_state_label desc, avg_speed_kmh asc

=== 2. Zone-level congestion summary (last 30 min) ===
TrafficStream
| where timestamp > ago(30m)
| lookup kind=leftouter (Segment) on segment_id
| summarize avg_speed_kmh    = round(avg(avg_speed_kmh), 1),
            total_vehicles   = sum(vehicle_count),
            pct_congested    = round(100.0 * countif(speed_state_code == 3) / count(), 1),
            incident_rate_pct = round(100.0 * countif(incident_flag) / count(), 1)
  by zone_approx
| order by pct_congested d